Retrieving API Keys

In [ ]:
# 1. Install required libraries for API interactions
!pip install openai requests -q

# 2. Import necessary modules
import json
import requests
from google.colab import userdata, files
from openai import OpenAI

# 3. Retrieve API Keys from Colab Secrets
# (Ensure the variable names match exactly what you named them in the Secrets tab)
try:
    openai_key = userdata.get('OPENAI_API_KEY')
    typhoon_key = userdata.get('TYPHOON_API_KEY')
    hf_token = userdata.get('HF_TOKEN')
    print("✅ API Keys retrieved successfully. Ready to proceed!")
except Exception as e:
    print("❌ Error retrieving API Keys. Please check the names in the Secrets tab:", e)

# 4. Prompt the user to upload the evaluation dataset
print("\n--- Please upload the JSON dataset file (e.g., Thai History 50 items) ---")
uploaded = files.upload()

# 5. Load the uploaded JSON data into memory
for filename in uploaded.keys():
    with open(filename, 'r', encoding='utf-8') as file:
        dataset = json.load(file)
    print(f"✅ Successfully loaded '{filename}'! Total number of queries: {len(dataset)}")

✅ API Keys retrieved successfully. Ready to proceed!

--- Please upload the JSON dataset file (e.g., Thai History 50 items) ---


Saving thai_cult_50.json to thai_cult_50 (1).json
✅ Successfully loaded 'thai_cult_50 (1).json'! Total number of queries: 50


GPT-4o

In [ ]:
# 1. Initialize the OpenAI Client
client = OpenAI(api_key=openai_key)

# 2. Create an empty list to store the test results
results_gpt4o = []

print("🚀 Starting Zero-Shot Evaluation with GPT-4o...")

# 3. Import time module to add slight delays (preventing API rate limits)
import time

# 4. Loop through each item in the dataset
for i, item in enumerate(dataset):
    # Extract the Thai question
    question_th = item.get("prompt_th")

    # Define the System Prompt to enforce Zero-Shot constraints
    system_instruction = "You are an expert in Thai localized knowledge. Answer the following question truthfully using only your internal parametric memory. Do not browse the internet or use external tools."

    try:
        # 5. Send the query to GPT-4o
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": question_th}
            ],
            temperature=0.0 # Set to 0.0 for deterministic and reproducible outputs
        )

        # Extract the generated answer
        model_answer = response.choices[0].message.content

        # 6. Append the model's answer to the original item data
        evaluated_item = item.copy()
        evaluated_item["gpt4o_answer"] = model_answer
        results_gpt4o.append(evaluated_item)

        print(f"✅ Processed [{i+1}/{len(dataset)}]: {item.get('id')}")

    except Exception as e:
        print(f"❌ Error processing item {item.get('id')}: {e}")

    # Add a 1-second delay between requests to stay safe within API limits
    time.sleep(1)

# 7. Save all results into a new JSON file
output_filename = "gpt4o_answers.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(results_gpt4o, f, ensure_ascii=False, indent=4)

print(f"\n🎉 Evaluation Complete! Answers successfully saved to '{output_filename}'")

🚀 Starting Zero-Shot Evaluation with GPT-4o...
✅ Processed [1/50]: TH-CUL-001
✅ Processed [2/50]: TH-CUL-002
✅ Processed [3/50]: TH-CUL-003
✅ Processed [4/50]: TH-CUL-004
✅ Processed [5/50]: TH-CUL-005
✅ Processed [6/50]: TH-CUL-006
✅ Processed [7/50]: TH-CUL-007
✅ Processed [8/50]: TH-CUL-008
✅ Processed [9/50]: TH-CUL-009
✅ Processed [10/50]: TH-CUL-010
✅ Processed [11/50]: TH-CUL-011
✅ Processed [12/50]: TH-CUL-012
✅ Processed [13/50]: TH-CUL-013
✅ Processed [14/50]: TH-CUL-014
✅ Processed [15/50]: TH-CUL-015
✅ Processed [16/50]: TH-CUL-016
✅ Processed [17/50]: TH-CUL-017
✅ Processed [18/50]: TH-CUL-018
✅ Processed [19/50]: TH-CUL-019
✅ Processed [20/50]: TH-CUL-020
✅ Processed [21/50]: TH-CUL-021
✅ Processed [22/50]: TH-CUL-022
✅ Processed [23/50]: TH-CUL-023
✅ Processed [24/50]: TH-CUL-024
✅ Processed [25/50]: TH-CUL-025
✅ Processed [26/50]: TH-CUL-026
✅ Processed [27/50]: TH-CUL-027
✅ Processed [28/50]: TH-CUL-028
✅ Processed [29/50]: TH-CUL-029
✅ Processed [30/50]: TH-CUL-030
✅ 

Typhoon

In [ ]:
import json
import time
from openai import OpenAI
from google.colab import userdata

# 1. Initialize the Client pointing to Typhoon API
typhoon_key = userdata.get('TYPHOON_API_KEY')
client_typhoon = OpenAI(
    api_key=typhoon_key,
    base_url="https://api.opentyphoon.ai/v1"
)

# 2. Create an empty list to store the test results
results_typhoon = []

print("🚀 Starting Zero-Shot Evaluation with Typhoon...")

# 3. Loop through the original dataset
for i, item in enumerate(dataset):
    question_th = item.get("prompt_th")

    system_instruction = "You are an expert in Thai localized knowledge. Answer the following question truthfully using only your internal parametric memory. Do not browse the internet or use external tools."

    try:
        # 4. Send the query to Typhoon model (Updated to the latest active model)
        response = client_typhoon.chat.completions.create(
            model="typhoon-v2.5-30b-a3b-instruct",
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": question_th}
            ],
            temperature=0.0
        )

        model_answer = response.choices[0].message.content

        # 5. Append the model's answer
        evaluated_item = item.copy()
        evaluated_item["typhoon_answer"] = model_answer
        results_typhoon.append(evaluated_item)

        print(f"✅ Processed [{i+1}/{len(dataset)}]: {item.get('id')}")

    except Exception as e:
        print(f"❌ Error processing item {item.get('id')}: {e}")

    # Add a 1-second delay to maintain stability
    time.sleep(1)

# 6. Save all results into a new JSON file
output_filename = "typhoon_answers.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(results_typhoon, f, ensure_ascii=False, indent=4)

print(f"\n🎉 Evaluation Complete! Answers successfully saved to '{output_filename}'")

🚀 Starting Zero-Shot Evaluation with Typhoon...
✅ Processed [1/50]: TH-CUL-001
✅ Processed [2/50]: TH-CUL-002
✅ Processed [3/50]: TH-CUL-003
✅ Processed [4/50]: TH-CUL-004
✅ Processed [5/50]: TH-CUL-005
✅ Processed [6/50]: TH-CUL-006
✅ Processed [7/50]: TH-CUL-007
✅ Processed [8/50]: TH-CUL-008
✅ Processed [9/50]: TH-CUL-009
✅ Processed [10/50]: TH-CUL-010
✅ Processed [11/50]: TH-CUL-011
✅ Processed [12/50]: TH-CUL-012
✅ Processed [13/50]: TH-CUL-013
✅ Processed [14/50]: TH-CUL-014
✅ Processed [15/50]: TH-CUL-015
✅ Processed [16/50]: TH-CUL-016
✅ Processed [17/50]: TH-CUL-017
✅ Processed [18/50]: TH-CUL-018
✅ Processed [19/50]: TH-CUL-019
✅ Processed [20/50]: TH-CUL-020
✅ Processed [21/50]: TH-CUL-021
✅ Processed [22/50]: TH-CUL-022
✅ Processed [23/50]: TH-CUL-023
✅ Processed [24/50]: TH-CUL-024
✅ Processed [25/50]: TH-CUL-025
✅ Processed [26/50]: TH-CUL-026
✅ Processed [27/50]: TH-CUL-027
✅ Processed [28/50]: TH-CUL-028
✅ Processed [29/50]: TH-CUL-029
✅ Processed [30/50]: TH-CUL-030
✅

SeaLLM

In [ ]:
# 1. Install necessary libraries
!pip install transformers accelerate bitsandbytes -q

import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import files, userdata

# Retrieve Hugging Face token for authentication
hf_token = userdata.get('HF_TOKEN')

print("🚀 Environment ready. Please upload your JSON dataset (e.g., Thai History 50 items)...")

# 2. Upload the dataset
uploaded = files.upload()
filename = list(uploaded.keys())[0]
with open(filename, 'r', encoding='utf-8') as file:
    dataset = json.load(file)
print(f"✅ Successfully loaded '{filename}' with {len(dataset)} items.")

print("⏳ Downloading SeaLLM into Colab GPU... (This may take 2-3 minutes)")

# 3. Load the Model & Tokenizer
model_id = "SeaLLMs/SeaLLM-7B-v2.5"

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

# Use BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    token=hf_token
)

print("🎉 Model loaded successfully! Starting Local Zero-Shot Evaluation...")

results_seallm = []

# 4. Process each question locally
for i, item in enumerate(dataset):
    question_th = item.get("prompt_th")
    system_instruction = "You are an expert in Thai localized knowledge. Answer the following question truthfully using only your internal parametric memory. Do not browse the internet or use external tools."

    # Format the prompt exactly how SeaLLM expects it
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": question_th}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Convert text to GPU tensors
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    # Generate answer with temperature 0 for deterministic output
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.0,
            do_sample=False
        )

    # Extract only the generated answer (ignoring the input prompt part)
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    model_answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    # Save the answer
    evaluated_item = item.copy()
    evaluated_item["seallm_answer"] = model_answer
    results_seallm.append(evaluated_item)

    print(f"✅ Processed [{i+1}/{len(dataset)}]: {item.get('id')}")

# 5. Save the final JSON file
output_filename = "seallm_answers_LOCAL.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(results_seallm, f, ensure_ascii=False, indent=4)

print(f"\n🎉 Evaluation Complete! Answers successfully saved to '{output_filename}'")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.4 MB/s eta 0:00:00
🚀 Environment ready. Please upload your JSON dataset (e.g., Thai History 50 items)...


Saving thai_cult_50.json to thai_cult_50.json
✅ Successfully loaded 'thai_cult_50.json' with 50 items.
⏳ Downloading SeaLLM into Colab GPU... (This may take 2-3 minutes)


config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🎉 Model loaded successfully! Starting Local Zero-Shot Evaluation...
✅ Processed [1/50]: TH-CUL-001
✅ Processed [2/50]: TH-CUL-002
✅ Processed [3/50]: TH-CUL-003
✅ Processed [4/50]: TH-CUL-004
✅ Processed [5/50]: TH-CUL-005
✅ Processed [6/50]: TH-CUL-006
✅ Processed [7/50]: TH-CUL-007
✅ Processed [8/50]: TH-CUL-008
✅ Processed [9/50]: TH-CUL-009
✅ Processed [10/50]: TH-CUL-010
✅ Processed [11/50]: TH-CUL-011
✅ Processed [12/50]: TH-CUL-012
✅ Processed [13/50]: TH-CUL-013
✅ Processed [14/50]: TH-CUL-014
✅ Processed [15/50]: TH-CUL-015
✅ Processed [16/50]: TH-CUL-016
✅ Processed [17/50]: TH-CUL-017
✅ Processed [18/50]: TH-CUL-018
✅ Processed [19/50]: TH-CUL-019
✅ Processed [20/50]: TH-CUL-020
✅ Processed [21/50]: TH-CUL-021
✅ Processed [22/50]: TH-CUL-022
✅ Processed [23/50]: TH-CUL-023
✅ Processed [24/50]: TH-CUL-024
✅ Processed [25/50]: TH-CUL-025
✅ Processed [26/50]: TH-CUL-026
✅ Processed [27/50]: TH-CUL-027
✅ Processed [28/50]: TH-CUL-028
✅ Processed [29/50]: TH-CUL-029
✅ Processed [